# 🧬 Mikrobioma Usus Bayi: Perbandingan Persalinan Vaginal vs Sesar dengan 16S Long-Read

**Modul hands-on Google Colab | OmicsLite**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmicsLite/Workshop-Long-Reads-Metagenomics/blob/main/hands-on/gut_microbiome_16s.ipynb)

> **Istilah penting:** praktikum ini menganalisis **amplicon gen 16S rRNA**, bukan *shotgun metagenomics*. Amplicon 16S menargetkan satu gen penanda untuk membuat profil taksonomi bakteri/arkaea; ia tidak mengukur seluruh gen di komunitas dan tidak dapat langsung menyimpulkan fungsi metabolik.

## Tujuan belajar

Setelah menyelesaikan notebook ini, peserta mampu:

1. menjelaskan isi FASTQ dan skor kualitas Phred;
2. menilai distribusi panjang dan kualitas read ONT;
3. menyaring read 16S long-read (V3–V9) secara reproducible;
4. mengklasifikasikan read dengan Kraken2 + NCBI Targeted Loci (16S/18S);
5. membuat tabel komposisi dan visualisasi Krona;
6. membandingkan alpha/beta diversity antara kelompok persalinan;
7. menjalankan uji kelompok eksploratif dan menafsirkan keterbatasannya.

## Mengapa 16S long-read?

Gen 16S rRNA (~1.500 bp) memiliki daerah lestari universal untuk primer PCR dan sembilan daerah variabel (V1–V9). Dataset praktikum ini (PRJNA637202; Angell et al., 2020) menargetkan wilayah V3–V9 (~1.140 bp) yang informatif secara taksonomi. ONT dapat membaca amplicon panjang, bahkan mendekati seluruh gen. Namun, read dataset ini berpusat sekitar 1,14 kb sehingga tidak semuanya full-length utuh. Resolusi umumnya lebih baik daripada hanya satu region pendek, tetapi **identifikasi spesies tidak otomatis akurat**: kualitas read, primer, database, kemiripan antarspesies, dan variasi jumlah kopi gen 16S tetap membatasi interpretasi.

## Alur praktikum

```text
ENA FASTQ → validasi → NanoPlot → Chopper (atau SeqKit) → Kraken2/NCBI-Targeted-Loci
                                           ├→ tabel komposisi
                                           ├→ Krona
                                           └→ rarefaction → alpha & beta diversity
```

**Estimasi:** unduhan 12 FASTQ ±510 MB, database ±8.6 MB, dan waktu total sekitar 25–45 menit pada Colab CPU (bergantung jaringan/solver). Jalankan cell berurutan dari atas. Runtime Colab bersifat sementara; unduh hasil sebelum sesi berakhir.

---
## 0. Konfigurasi praktikum

Semua parameter penting dikumpulkan di satu tempat. Untuk workshop, jangan mengubahnya sebelum pipeline berhasil sekali. `SEED` memastikan rarefaction dapat direproduksi.

In [ ]:
# @title 0.1 Parameter analisis & struktur folder terorganisir
from pathlib import Path

WORKDIR = Path("/content/16s_workshop") if Path("/content").exists() else Path.cwd() / "16s_workshop"
RAW_DIR = WORKDIR / "raw"
FILTERED_DIR = WORKDIR / "filtered"
QC_DIR = WORKDIR / "qc"
DB_DIR = WORKDIR / "database_16s"

# Struktur folder hasil terorganisir per tahap analisis (traceability & reproducibility)
RESULTS_DIR = WORKDIR / "results"
RES_QC_DIR = RESULTS_DIR / "01_qc"
RES_CLASS_DIR = RESULTS_DIR / "02_classification"
RES_CLASS_REPORTS_DIR = RES_CLASS_DIR / "reports"
RES_CLASS_PER_READ_DIR = RES_CLASS_DIR / "per_read"
RES_ABUND_TABLES_DIR = RESULTS_DIR / "03_abundance" / "tables"
RES_ABUND_FIGS_DIR = RESULTS_DIR / "03_abundance" / "figures"
RES_KRONA_HTML_DIR = RESULTS_DIR / "04_krona" / "html"
RES_KRONA_TSV_DIR = RESULTS_DIR / "04_krona" / "tsv"
RES_ALPHA_DIR = RESULTS_DIR / "05_diversity" / "alpha"
RES_BETA_DIR = RESULTS_DIR / "05_diversity" / "beta"
RES_DIFF_DIR = RESULTS_DIR / "06_differential_abundance"

MIN_LENGTH = 1000
MAX_LENGTH = 1700
MIN_MEAN_Q = 10
KRAKEN_CONFIDENCE = 0.01
THREADS = 2
SEED = 42

ALL_DIRS = (
    RAW_DIR, FILTERED_DIR, QC_DIR, DB_DIR,
    RES_QC_DIR, RES_CLASS_REPORTS_DIR, RES_CLASS_PER_READ_DIR,
    RES_ABUND_TABLES_DIR, RES_ABUND_FIGS_DIR,
    RES_KRONA_HTML_DIR, RES_KRONA_TSV_DIR,
    RES_ALPHA_DIR, RES_BETA_DIR, RES_DIFF_DIR,
)
for directory in ALL_DIRS:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Workspace : {WORKDIR}")
print(f"Hasil     : {RESULTS_DIR} (terbagi ke subfolder 01_qc s.d. 06_differential_abundance)")
print(f"Filter    : {MIN_LENGTH}–{MAX_LENGTH} bp; mean Q ≥ {MIN_MEAN_Q}")


---
## 1. Instalasi alat bioinformatika

Notebook memakai **micromamba** agar alat command-line terisolasi dari Python bawaan Colab dan runtime tidak perlu restart. Versi alat ditampilkan setelah instalasi untuk jejak reproducibility.

In [ ]:
# @title 1.1 Instal Chopper, Kraken2, Krona, NanoPlot, dan SeqKit
import os
import shutil
import subprocess
from pathlib import Path

MICROMAMBA = (Path("/content") if Path("/content").exists() else WORKDIR) / "micromamba"
ENV_PREFIX = Path("/content/bioenv") if Path("/content").exists() else WORKDIR / "bioenv"
REQUIRED_TOOLS = ("chopper", "kraken2", "ktImportText", "NanoPlot", "seqkit")

def run(cmd, *, stdout=None, cwd=None):
    """Jalankan command dan hentikan notebook segera bila command gagal."""
    printable = " ".join(map(str, cmd))
    print(f"$ {printable}")
    return subprocess.run(list(map(str, cmd)), check=True, stdout=stdout, cwd=cwd)

if not all(shutil.which(cmd) for cmd in REQUIRED_TOOLS):
    if not MICROMAMBA.exists():
        run(["bash", "-lc", "curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /tmp bin/micromamba"])
        shutil.copy2("/tmp/bin/micromamba", MICROMAMBA)
        MICROMAMBA.chmod(0o755)

    if not (ENV_PREFIX / "bin" / "kraken2").exists():
        run([
            MICROMAMBA, "create", "-y", "-p", ENV_PREFIX,
            "--override-channels", "-c", "conda-forge", "-c", "bioconda",
            "--strict-channel-priority",
            "python=3.11", "chopper>=0.8", "kraken2>=2.1", "krona>=2.8",
            "nanoplot>=1.40", "seqkit>=2.8",
        ])

    os.environ["PATH"] = f"{ENV_PREFIX / 'bin'}:{os.environ['PATH']}"
else:
    print("ℹ️ Semua alat bioinformatika sudah tersedia di PATH lokal; instalasi micromamba dilewati.")

for command in REQUIRED_TOOLS:
    assert shutil.which(command), f"{command} tidak ditemukan di PATH"

run(["kraken2", "--version"])
run(["chopper", "--version"])
run(["seqkit", "version"])
print("\n✅ Semua alat siap.")


### Sekilas tentang pasangan `Kraken2` → `Bracken` (dan kenapa di sini kita pakai `Kraken2` langsung)

Di banyak tutorial dan pipeline metagenomik (termasuk *workflow* resmi EPI2ME), `Kraken2` sering dipasangkan dengan alat pendamping bernama **Bracken** (*Bayesian Reestimation of Abundance with Kraken*):

```text
FASTQ terfilter  →  Kraken2 (klasifikasi k-mer + LCA)  →  [Opsional] Bracken (pembagian ulang kelimpahan)
```

* **Apa fungsi Bracken?** Saat `Kraken2` menemukan *read* yang ambigu antarspesies dekat, ia menahannya di tingkat atas (Family atau Genus). Bracken bertugas mengambil *read* yang tertahan di tingkat atas itu, lalu membaginya ulang secara probabilistik ke spesies-spesies di bawahnya.
* **Kenapa kita singgung di sini?** Di Sel 1.2 tepat di bawah ini, saat mengunduh database ONT EPI2ME `ncbi_16s_18s`, Anda akan melihat file `database1000mers.kmer_distrib`—itulah file model Bracken untuk panjang bacaan ~1.000 bp.

**Kenapa pada praktikum ini kita memakai laporan `Kraken2` langsung tanpa Bracken?**
1. **Menghindari pembesaran *false positive* pada data R9.4.1**: dengan tingkat error *indel* ~10%, jika muncul spesies keliru akibat *noise* k-mer, Bracken berisiko mengalirkan ribuan *read* dari tingkat Genus/Family ke spesies yang salah tersebut.
2. **Lebih transparan untuk belajar konsep LCA**: kita bisa melihat apa adanya *read* mana yang cukup yakin turun ke Spesies dan mana yang ditahan di tingkat Genus/Family.

Bracken lebih tepat dipakai pada data *shotgun* atau Nanopore generasi baru **R10.4.1 (Q20+)**. Sementara itu, jika pada riset mandiri Anda ingin akurasi spesies tertinggi dari data 16S Nanopore, coba evaluasi **`Emu`** (berbasis *Expectation-Maximization*).

In [ ]:
# @title 1.2 Unduh dan validasi database NCBI Targeted Loci 16S/18S (ONT EPI2ME)
import hashlib

DB_URL = "https://s3.eu-west-1.amazonaws.com/ont-open-data/workflow-databases/wf-metagenomics-dbs/ncbi_16s_18s/ncbi_targeted_loci_kraken2.tar.gz"
DB_ARCHIVE = WORKDIR / "ncbi_targeted_loci_kraken2.tar.gz"
EXPECTED_SHA256 = "f11135cdafebc20d179032b50a72b1ccc5a60b959f76b3f1b6b3198e67ca6aae"

required_db_files = [DB_DIR / name for name in ("hash.k2d", "opts.k2d", "taxo.k2d")]
if not all(path.exists() for path in required_db_files):
    if not DB_ARCHIVE.exists() or DB_ARCHIVE.stat().st_size == 0:
        run(["wget", "--continue", "--tries=3", "--timeout=30", "-O", DB_ARCHIVE, DB_URL])
    
    # Validasi integritas SHA256
    with open(DB_ARCHIVE, "rb") as f:
        file_hash = hashlib.sha256(f.read()).hexdigest()
    if file_hash == EXPECTED_SHA256:
        print(f"🔒 Verifikasi SHA256 valid: {file_hash[:16]}...")
    else:
        print(f"⚠️ Peringatan: Hash SHA256 berbeda ({file_hash[:16]}... vs {EXPECTED_SHA256[:16]}...)")
        
    run(["tar", "-xzf", DB_ARCHIVE, "-C", DB_DIR])

missing = [str(path) for path in required_db_files if not path.exists()]
assert not missing, f"Database tidak lengkap: {missing}"

bracken_models = sorted(path.name for path in DB_DIR.glob("database*mers.kmer_distrib"))
print("✅ Database Kraken2 NCBI Targeted Loci valid:", DB_DIR)
print("Model panjang read Bracken yang tersedia:", ", ".join(bracken_models) if bracken_models else "None")

---
## 2. Data publik, metadata, dan desain kelompok

BioProject **PRJNA637202** berisi 96 sampel feses bayi berusia 3 bulan dari studi PreventADALL. ENA mencatat data sebagai `AMPLICON`, `PCR`, `SINGLE`, platform `OXFORD_NANOPORE`, dan instrumen `MinION`.

File suplementer `MetadataPublish.xlsx` menghubungkan barcode ONT dengan mode persalinan: `0 = vaginal` dan `1 = sectio/sesar`. Barcode `BCxx` cocok dengan alias ENA `MergedSeqxx`, yang kemudian dipetakan ke accession SRR.

Untuk menjaga durasi workshop, kita memakai **subset seimbang 6 vaginal + 6 sesar**. Pemilihan hanya berdasarkan kecukupan jumlah read mentah, bukan berdasarkan komposisi mikrobioma. Ini adalah desain didaktik dan tidak dimaksudkan mereproduksi seluruh analisis publikasi.

> Mode persalinan adalah variabel kelompok, bukan diagnosis. Tanpa kovariat lengkap (misalnya antibiotik, pola menyusui, lokasi, dan faktor maternal), perbedaan yang ditemukan tetap berupa asosiasi eksploratif.


In [ ]:
# @title 2.1 Manifest subset seimbang dan unduh FASTQ (GitHub Release → Fallback Paralel ENA)
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
from pathlib import Path

# Tautan arsip 12 FASTQ di GitHub Releases (jauh lebih cepat ~10 dtk di Colab jika sudah diunggah & repo Public)
RELEASE_ARCHIVE_URL = (
    "https://github.com/OmicsLite/Workshop-Long-Reads-Metagenomics/"
    "releases/download/v1.0-workshop-data/prjna637202_12samples_raw.tar.gz"
)

manifest_file = Path("sample_manifest.tsv")
if not manifest_file.exists():
    manifest_file = Path("hands-on/sample_manifest.tsv")

if manifest_file.exists():
    samples = pd.read_csv(manifest_file, sep="\t")
else:
    samples = pd.DataFrame([
        {"sample": "V01", "group": "Vaginal", "birth_code": 0, "barcode": "BC63", "alias": "MergedSeq63", "run": "SRR11923954", "read_count_ena": 45278, "bytes": 47935767, "subdir": "054"},
        {"sample": "V02", "group": "Vaginal", "birth_code": 0, "barcode": "BC73", "alias": "MergedSeq73", "run": "SRR11923943", "read_count_ena": 44293, "bytes": 47062268, "subdir": "043"},
        {"sample": "V03", "group": "Vaginal", "birth_code": 0, "barcode": "BC83", "alias": "MergedSeq83", "run": "SRR11923932", "read_count_ena": 43150, "bytes": 46042841, "subdir": "032"},
        {"sample": "V04", "group": "Vaginal", "birth_code": 0, "barcode": "BC76", "alias": "MergedSeq76", "run": "SRR11923940", "read_count_ena": 42560, "bytes": 45295782, "subdir": "040"},
        {"sample": "V05", "group": "Vaginal", "birth_code": 0, "barcode": "BC46", "alias": "MergedSeq46", "run": "SRR11923973", "read_count_ena": 42535, "bytes": 44492008, "subdir": "073"},
        {"sample": "V06", "group": "Vaginal", "birth_code": 0, "barcode": "BC77", "alias": "MergedSeq77", "run": "SRR11923939", "read_count_ena": 42091, "bytes": 43781623, "subdir": "039"},
        {"sample": "C01", "group": "Sesar", "birth_code": 1, "barcode": "BC49", "alias": "MergedSeq49", "run": "SRR11923970", "read_count_ena": 43527, "bytes": 45463513, "subdir": "070"},
        {"sample": "C02", "group": "Sesar", "birth_code": 1, "barcode": "BC89", "alias": "MergedSeq89", "run": "SRR11923926", "read_count_ena": 39632, "bytes": 40936549, "subdir": "026"},
        {"sample": "C03", "group": "Sesar", "birth_code": 1, "barcode": "BC19", "alias": "MergedSeq19", "run": "SRR11924003", "read_count_ena": 37036, "bytes": 38865995, "subdir": "003"},
        {"sample": "C04", "group": "Sesar", "birth_code": 1, "barcode": "BC84", "alias": "MergedSeq84", "run": "SRR11923931", "read_count_ena": 36843, "bytes": 38789861, "subdir": "031"},
        {"sample": "C05", "group": "Sesar", "birth_code": 1, "barcode": "BC86", "alias": "MergedSeq86", "run": "SRR11923929", "read_count_ena": 35293, "bytes": 36288277, "subdir": "029"},
        {"sample": "C06", "group": "Sesar", "birth_code": 1, "barcode": "BC66", "alias": "MergedSeq66", "run": "SRR11923951", "read_count_ena": 34498, "bytes": 36412478, "subdir": "051"},
    ])

samples["url"] = samples.apply(
    lambda row: f"https://ftp.sra.ebi.ac.uk/vol1/fastq/{row.run[:6]}/{str(row['subdir']).zfill(3)}/{row.run}/{row.run}_1.fastq.gz",
    axis=1,
)
samples["raw_fastq"] = samples["run"].map(lambda run_id: str(RAW_DIR / f"{run_id}.fastq.gz"))

assert samples.groupby("group").size().nunique() == 1, "Kelompok tidak seimbang"
assert samples["run"].is_unique and samples["barcode"].is_unique
display(samples[["sample", "group", "barcode", "alias", "run", "read_count_ena"]])

def all_fastq_valid():
    return all(
        Path(row.raw_fastq).exists() and Path(row.raw_fastq).stat().st_size > 10_000_000
        for row in samples.itertuples(index=False)
    )

# Lapis 1: Coba unduh 1 arsip cepat dari GitHub Releases
if not all_fastq_valid() and RELEASE_ARCHIVE_URL:
    release_tar = WORKDIR / "prjna637202_12samples_raw.tar.gz"
    print("🚀 Mencoba unduh cepat dari GitHub Releases...")
    res = subprocess.run(
        ["wget", "--continue", "--tries=2", "--timeout=30", "-q", "-O", str(release_tar), RELEASE_ARCHIVE_URL]
    )
    if res.returncode == 0 and release_tar.exists() and release_tar.stat().st_size > 1_000_000:
        print("📦 Mengekstrak arsip GitHub Release ke folder raw/...")
        run(["tar", "-xzf", release_tar, "-C", RAW_DIR])
    else:
        if release_tar.exists():
            release_tar.unlink()
        print("ℹ️ Arsip GitHub Release belum tersedia; otomatis beralih ke unduhan paralel dari ENA...")

# Lapis 2 (Fallback): Unduh paralel dari ENA + verifikasi ukuran byte eksak
def download_one(row):
    target = Path(row.raw_fastq)
    expected = int(row.bytes)
    if target.exists() and target.stat().st_size > 10_000_000:
        return row.sample, target.stat().st_size, "ready"
    subprocess.run(
        ["wget", "--continue", "--tries=5", "--timeout=45", "-q", "-O", str(target), row.url],
        check=True,
    )
    actual = target.stat().st_size
    assert actual == expected, f"Ukuran {row.sample} tidak cocok ({actual} vs {expected} bytes); jalankan ulang sel ini."
    return row.sample, actual, "downloaded"

if not all_fastq_valid():
    print("⬇️ Mengunduh FASTQ yang belum lengkap secara paralel (4 koneksi bersamaan)...")
done_count = 0
with ThreadPoolExecutor(max_workers=4) as pool:
    futures = [pool.submit(download_one, row) for row in samples.itertuples(index=False)]
    for fut in as_completed(futures):
        sample_id, size_b, status = fut.result()
        done_count += 1
        print(f"  [{done_count:02d}/{len(samples)}] ✅ {sample_id} ({size_b / 1e6:.1f} MB) — {status}")

print(f"\n✅ {len(samples)} FASTQ terunduh utuh & terverifikasi; {samples.groupby('group').size().to_dict()}")


In [ ]:
# @title 2.2 Validasi integritas FASTQ dan lihat contoh satu read
stats_raw_path = RES_QC_DIR / "seqkit_stats_raw.tsv"
with stats_raw_path.open("w") as handle:
    run(["seqkit", "stats", "-T", *samples["raw_fastq"]], stdout=handle)

stats_raw = pd.read_csv(stats_raw_path, sep="\t")
display(stats_raw)

# FASTQ terdiri dari 4 baris per read: header, sekuens, separator '+', dan kualitas.
import gzip
with gzip.open(samples.loc[0, "raw_fastq"], "rt") as handle:
    header, sequence, separator, quality = [handle.readline().strip() for _ in range(4)]
print("Header       :", header)
print("Panjang read :", len(sequence), "bp")
print("Sekuens awal :", sequence[:80] + "...")
print("Kualitas awal:", quality[:80] + "...")
assert separator == "+" and len(sequence) == len(quality)
print("\n✅ SeqKit dapat membaca semua file; struktur FASTQ valid.")

### Catatan baca output: isi 4 baris FASTQ & cara cepat menghafal Q-Score

Satu *read* di file FASTQ selalu terdiri dari **4 baris**: header ID (`@`), urutan DNA (`A, C, G, T`), pemisah `+`, dan baris kualitas dalam karakter ASCII (*Phred+33*).

* **Phred Q-Score** mengukur peluang salah baca per basa: $Q = -10 \cdot \log_{10}(P_{\text{error}})$.
* **Cara cepat menghafalnya tanpa menghitung logaritma**: tiap nilai $Q$ naik **10**, peluang error turun **10 kali lipat**:
  * **Q10** (`+`) = 1 salah dari 10 basa (akurasi 90%)
  * **Q20** (`5`) = 1 salah dari 100 basa (akurasi 99%)
  * **Q30** (`?`) = 1 salah dari 1.000 basa (akurasi 99,9%)

---
## 3. Quality control dan filtering

Skor Phred menghubungkan kualitas dengan probabilitas kesalahan: $Q=-10\log_{10}(P_{error})$. Q10 berarti estimasi error 10% per basa, bukan jaminan bahwa seluruh read 90% identik terhadap referensi.

Berdasarkan distribusi dataset ini (median sekitar 1,14 kb), filter workshop mempertahankan read 1.000–1.700 bp dengan mean Q≥10. Memaksakan cutoff 1.300 bp akan membuang hampir seluruh data. Rentang ini tetap **bukan detektor chimera**. Chimera PCR dapat memiliki panjang normal; analisis riset perlu langkah deteksi chimera/denoising yang tervalidasi untuk data ONT.

In [ ]:
# @title 3.1 Buat dan tampilkan laporan NanoPlot untuk sampel contoh (V01)
from IPython.display import HTML, display

nanoplot_dir = QC_DIR / "V01_raw"
report = nanoplot_dir / "NanoPlot-report.html"
if not report.exists():
    run([
        "NanoPlot", "--fastq", samples.loc[0, "raw_fastq"],
        "--outdir", nanoplot_dir, "--plots", "dot", "--threads", str(THREADS),
    ])

display(HTML(filename=str(report)))
shutil.copy2(report, RES_QC_DIR / "V01_NanoPlot-report.html")
print("Jika iframe tidak tampil otomatis di browser, buka dari panel Files:", report)

### Cara membaca laporan NanoPlot di atas

Ada dua hal utama yang perlu dicek dari grafik distribusi panjang dan kualitas tersebut:
1. **Puncak panjang read berada di ~1.140 bp (1,14 kb)**. Ini bukan karena DNA rusak atau terpotong. Primer PCR pada studi PRJNA637202 memang menargetkan wilayah **V3–V9** (~1,14 kb), bukan full-length V1–V9 (~1,5 kb).
2. **Kualitas rata-rata di kisaran Q10–Q14**. Angka ini wajar untuk kimia Nanopore **R9.4.1** keluaran tahun 2020 (sebagai perbandingan, kimia modern **R10.4.1** kini sudah mencapai >Q20). Karena itu, batas filter kita di sel berikutnya diatur di **Q ≥ 10** dan **1.000–1.700 bp**.

In [ ]:
# @title 3.2 Filter semua sampel dengan Chopper
filtered_paths = []
for row in samples.itertuples(index=False):
    output = FILTERED_DIR / f"{row.run}.filtered_L{MIN_LENGTH}-{MAX_LENGTH}_Q{MIN_MEAN_Q}.fastq.gz"
    filtered_paths.append(str(output))
    if not output.exists() or output.stat().st_size == 0:
        command = (
            f"gunzip -c {row.raw_fastq} | "
            f"chopper -q {MIN_MEAN_Q} -l {MIN_LENGTH} --maxlength {MAX_LENGTH} | "
            f"gzip -c > {output}"
        )
        run(["bash", "-o", "pipefail", "-c", command])
    assert output.stat().st_size > 0, f"Tidak ada read lolos untuk {row.run}"

samples["filtered_fastq"] = filtered_paths

stats_filtered_path = RES_QC_DIR / "seqkit_stats_filtered.tsv"
with stats_filtered_path.open("w") as handle:
    run(["seqkit", "stats", "-T", *samples["filtered_fastq"]], stdout=handle)
stats_filtered = pd.read_csv(stats_filtered_path, sep="\t")

qc_summary = samples[["sample", "group", "run"]].copy()
qc_summary["raw_reads"] = stats_raw["num_seqs"].to_numpy()
qc_summary["filtered_reads"] = stats_filtered["num_seqs"].to_numpy()
qc_summary["retained_pct"] = 100 * qc_summary["filtered_reads"] / qc_summary["raw_reads"]
qc_summary.to_csv(RES_QC_DIR / "qc_summary.tsv", sep="\t", index=False)
display(qc_summary.style.format({"retained_pct": "{:.1f}%"}))
print("✅ Filtering selesai. Retensi sangat rendah adalah sinyal untuk memeriksa plot panjang/kualitas, bukan sekadar melanjutkan.")

### Memahami hasil filtering (`Chopper`) dan mitos chimera

`Chopper` bekerja seperti ayakan biji kopi: hanya meloloskan read dengan ukuran pas (`1.000–1.700 bp`) dan kualitas cukup (`Q ≥ 10`).
* Retensi sekitar **70–85%** tergolong sehat untuk data R9.4.1. Jika batas bawah dipaksa ke `1.300 bp`, lebih dari 90% data valid akan ikut terbuang.
* **Catatan penting tentang PCR chimera**: filter panjang memang membuang fragmen terlalu pendek (*primer-dimer*) dan terlalu panjang (*concatemer*), tetapi **tidak bisa menyaring chimera PCR**. Banyak molekul chimera punya panjang yang persis normal (~1,1–1,5 kb) sehingga tetap lolos filter ukuran.

### Checkpoint diskusi

- Apakah puncak panjang read berada dekat 1.500 bp?
- Berapa persen read yang hilang setelah filter, dan parameter mana yang mungkin paling berpengaruh?
- Apa trade-off jika `MIN_MEAN_Q` dinaikkan?

> Filtering dapat mengubah komposisi bila kualitas berbeda antar takson. Karena itu, simpan tabel retensi dan laporkan parameter yang dipakai.

---
## 4. Klasifikasi taksonomi dengan Kraken2

Kraken2 memetakan *minimizer* dari k-mer read ke takson dan menggunakan lowest common ancestor (LCA) ketika bukti tidak unik. Parameter `--confidence 0.01` mensyaratkan sedikitnya 1% k-mer informatif mendukung takson/keturunannya; ini secara efektif mengurangi panggilan acak (*spurious assignments*).

Database **NCBI Targeted Loci 16S/18S** dari ONT EPI2ME Labs dibangun dari strain tipe (*type strain*) resmi NCBI, memungkinkan identifikasi taksonomi hingga tingkat **Spesies** (`rank = S`) di samping tingkat **Genus** (`rank = G`).

> **Catatan Ilmiah Penting:** Kehadiran label taksonomi tingkat spesies pada laporan Kraken2 **bukan jaminan mutlak 100% kepastian spesies**. Beberapa spesies berkerabat dekat (misal dalam genus *Bacillus* atau *Escherichia/Shigella*) memiliki sekuens 16S yang nyaris identik. Oleh karena itu, kita menganalisis data pada **tingkat Spesies** sekaligus menyediakan rekapitulasi pada **tingkat Genus** sebagai kontrol metodologis.

In [ ]:
# @title 4.1 Jalankan Kraken2 dan simpan output per-read + report
kraken_outputs, kraken_reports = [], []

for row in samples.itertuples(index=False):
    output = RES_CLASS_PER_READ_DIR / f"{row.run}.kraken.tsv"
    report = RES_CLASS_REPORTS_DIR / f"{row.run}.k2report"
    kraken_outputs.append(str(output))
    kraken_reports.append(str(report))
    run([
        "kraken2", "--db", DB_DIR, "--threads", str(THREADS),
        "--confidence", str(KRAKEN_CONFIDENCE),
        "--output", output, "--report", report, row.filtered_fastq,
    ])
    assert report.exists() and report.stat().st_size > 0, f"Report gagal: {report}"

samples["kraken_output"] = kraken_outputs
samples["kraken_report"] = kraken_reports
print("✅ Klasifikasi selesai.")

In [ ]:
# @title 4.2 Ringkas classified vs unclassified
def read_kraken_report(path):
    columns = ["percent", "clade_reads", "direct_reads", "rank", "taxid", "name"]
    df = pd.read_csv(path, sep="\t", names=columns)
    df["name"] = df["name"].str.strip()
    return df

classification_rows = []
for row in samples.itertuples(index=False):
    report = read_kraken_report(row.kraken_report)
    unclassified = int(report.loc[report["rank"] == "U", "clade_reads"].sum())
    classified = int(report.loc[report["rank"] == "R", "clade_reads"].sum())
    total = classified + unclassified
    classification_rows.append({
        "sample": row.sample, "group": row.group, "run": row.run, "classified": classified,
        "unclassified": unclassified, "classified_pct": 100 * classified / total,
    })

classification_summary = pd.DataFrame(classification_rows)
classification_summary.to_csv(RES_CLASS_DIR / "classification_summary.tsv", sep="\t", index=False)
display(classification_summary.style.format({"classified_pct": "{:.1f}%"}))

### Cara kerja Kraken2: k-mer dan Lowest Common Ancestor (LCA)

`Kraken2` memotong tiap read menjadi fragmen pendek sepanjang 35 basa (*k-mer*) lalu mencocokkannya ke tabel hash database **ONT EPI2ME NCBI Targeted Loci** (~8,6 MB).
* **Prinsip LCA**: bayangkan detektif kamus yang tidak mau asal menebak. Kalau potongan k-mer cocok unik dengan satu spesies, read ditempatkan di **Spesies (`S`)**. Kalau cocok dengan beberapa spesies yang berkerabat dekat, Kraken2 mundur ke tingkat bersama terdekat (**Genus `G`** atau **Family `F`**).
* Opsi `--confidence 0.01` mewajibkan minimal 1% k-mer informatif mendukung cabang tersebut agar tebakan acak berkurang, tanpa membuang terlalu banyak sinyal pada data R9.4.1.

---
## 5. Tabel kelimpahan taksonomi (Spesies & Genus)

Untuk setiap takson (baik Spesies `rank = S` maupun Genus `rank = G`), nilai `clade_reads` mencakup seluruh read yang ditempatkan pada takson tersebut maupun varian turunannya. 

Kelimpahan relatif dinormalisasi terhadap total hitungan takson yang terdeteksi di sampel. Ini adalah **profil komposisional**: nilai persentase mencerminkan proporsi sekuens di dalam pustaka, bukan konsentrasi sel absolut di usus.

Untuk menyingkirkan panggilan tingkat rendah (*background noise*), kita menerapkan ambang $\ge 10$ reads per sampel untuk matriks utama **Genus**, serta $\ge 1$ read untuk eksplorasi **Spesies terarah**.

In [ ]:
# @title 5.1 Bentuk matriks hitungan taksonomi (Genus & Spesies)
import numpy as np
MIN_GENUS_READS = 10
MIN_SPECIES_READS = 1

def build_abundance_matrix(samples_df, target_rank="G", min_reads=10):
    series_dict = {}
    for row in samples_df.itertuples(index=False):
        report = read_kraken_report(row.kraken_report)
        subset = report.loc[report["rank"] == target_rank, ["name", "clade_reads"]].copy()
        subset = subset[subset["clade_reads"] >= min_reads]
        series_dict[row.sample] = subset.groupby("name")["clade_reads"].sum()
    counts = pd.DataFrame(series_dict).reindex(columns=samples_df["sample"]).fillna(0).astype(int).sort_index()
    return counts

# Matriks Utama (Kedalaman Tinggi): Tingkat Genus (rank = G)
genus_counts = build_abundance_matrix(samples, target_rank="G", min_reads=MIN_GENUS_READS)
genus_counts.to_csv(RES_ABUND_TABLES_DIR / "genus_counts.tsv", sep="\t")

genus_relative = genus_counts.div(genus_counts.sum(axis=0), axis=1).fillna(0)
genus_relative.to_csv(RES_ABUND_TABLES_DIR / "genus_relative_abundance.tsv", sep="\t")

# Matriks Eksploratif: Tingkat Spesies (rank = S)
species_counts = build_abundance_matrix(samples, target_rank="S", min_reads=MIN_SPECIES_READS)
species_counts.to_csv(RES_ABUND_TABLES_DIR / "species_counts.tsv", sep="\t")

species_relative = species_counts.div(species_counts.sum(axis=0), axis=1).fillna(0)
species_relative.to_csv(RES_ABUND_TABLES_DIR / "species_relative_abundance.tsv", sep="\t")

# Diagnostik kedalaman Genus vs Spesies (menunjukkan perilaku LCA Kraken2 pada R9.4.1)
depth_summary = pd.DataFrame({
    "group": samples.set_index("sample")["group"],
    "genus_reads": genus_counts.sum(axis=0),
    "species_reads": species_counts.sum(axis=0),
})
depth_summary["species_to_genus_pct"] = (100 * depth_summary["species_reads"] / depth_summary["genus_reads"].replace(0, np.nan)).round(1)
print(f"📊 Takson terdeteksi:")
print(f"   • Genus (rank G, ≥{MIN_GENUS_READS} read): {genus_counts.shape[0]} genus unik")
print(f"   • Spesies terarah (rank S, ≥{MIN_SPECIES_READS} read): {species_counts.shape[0]} spesies unik\n")
depth_summary.to_csv(RES_ABUND_TABLES_DIR / "lca_depth_summary.tsv", sep="\t")
display(depth_summary)


### Kenapa jumlah read Spesies lebih sedikit daripada Genus?

Lihat kolom `species_reads` dan `genus_reads` di tabel atas. Di hampir semua sampel, jumlah read yang sampai ke tingkat Spesies lebih kecil daripada tingkat Genus (sekitar 34–75% dari read Genus).

Ini hal yang normal pada data R9.4.1. Error *indel* homopolimer dan kemiripan sekuens 16S antarspesies yang berkerabat dekat (seperti *E. coli* dan *Shigella*) membuat Kraken2 menahan sebagian read di tingkat Genus agar tidak salah label. Untuk penelitian lanjutan yang mengejar identifikasi spesies seakurat mungkin, Anda bisa memakai **`Emu`** (Curry et al., 2022).

In [ ]:
# @title 5.2 Visualisasi komposisi taksonomi dominan (Genus & Spesies)
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
GROUP_COLORS = {"Vaginal": "#4C78A8", "Sesar": "#E45756"}
TOP_N = 12

def plot_stacked_composition(counts_df, rel_df, rank_label, out_filename, top_n=12):
    if counts_df.empty:
        print(f"⚠️ Tidak ada data untuk {rank_label}.")
        return
    top_taxa = counts_df.sum(axis=1).nlargest(top_n).index
    plot_table = rel_df.loc[top_taxa].T.copy()
    plot_table[f"{rank_label} lain"] = (1 - plot_table.sum(axis=1)).clip(lower=0)

    ax = (100 * plot_table).plot(kind="bar", stacked=True, figsize=(14, 5.5), colormap="tab20")
    ax.set_ylabel("Kelimpahan relatif (%)")
    ax.set_xlabel("Sampel (V = Vaginal; C = Sesar)")
    ax.set_title(f"Komposisi {min(top_n, len(top_taxa))} {rank_label} Dominan (NCBI Targeted Loci)")
    ax.legend(title=rank_label, bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.tick_params(axis="x", rotation=0)
    plt.tight_layout()
    plt.savefig(RES_ABUND_FIGS_DIR / out_filename, dpi=300)
    plt.show()

# 1. Grafik Utama: Tingkat Genus (kedalaman tinggi & stabil lintas semua sampel)
plot_stacked_composition(genus_counts, genus_relative, "Genus", "genus_composition_barplot.png", top_n=TOP_N)

# 2. Grafik Eksploratif: Tingkat Spesies Terarah (putative species)
plot_stacked_composition(species_counts, species_relative, "Spesies", "species_composition_barplot.png", top_n=TOP_N)


### Cara membaca grafik batang dan sifat data komposisional

Grafik batang bertumpuk di atas menampilkan **kelimpahan relatif (total selalu 100%)**, bukan jumlah sel bakteri sesungguhnya.

Bayangkan potongan kue pizza yang totalnya selalu 100%. Jika satu bakteri (misalnya *Bacteroides*) turun drastis, persentase bakteri lain (seperti *Bifidobacterium*) akan terlihat melonjak di grafik, padahal jumlah sel aslinya bisa saja tetap atau bahkan ikut turun. Untuk mengukur jumlah sel absolut di usus, kita butuh metode lab basah tambahan seperti *qPCR* 16S atau *flow cytometry*.

---
## 6. Krona interaktif

Utilitas **Krona Plot** merender hierarki taksonomi dalam diagram lingkaran konsentris bertingkat (*multi-layered pie chart*). Kita mengekstrak jalur taksonomi dari laporan Kraken2 NCBI Targeted Loci menjadi format input `ktImportText`. Diagram ini memungkinkan eksplorasi intuitif dari tingkat Domain $\rightarrow$ Phylum $\rightarrow$ Class $\rightarrow$ Order $\rightarrow$ Family $\rightarrow$ Genus $\rightarrow$ Species secara interaktif di browser.

In [ ]:
# @title 6.1 Buat Krona dari hierarki taksonomi NCBI
def kraken_report_to_krona_text(report_path, output_path):
    columns = ["percent", "clade_reads", "direct_reads", "rank", "taxid", "name"]
    report = pd.read_csv(report_path, sep="\t", names=columns, keep_default_na=False)
    lineage = {}
    with Path(output_path).open("w") as handle:
        for item in report.itertuples(index=False):
            raw_name = str(item.name)
            depth = (len(raw_name) - len(raw_name.lstrip())) // 2
            lineage[depth] = raw_name.strip()
            lineage = {level: name for level, name in lineage.items() if level <= depth}
            if item.rank not in ("U", "R") and int(item.direct_reads) > 0:
                path = [lineage[level] for level in sorted(lineage) if lineage[level] != "root"]
                handle.write(str(int(item.direct_reads)) + "\t" + "\t".join(path) + "\n")

krona_files = []
for row in samples.itertuples(index=False):
    krona_input = RES_KRONA_TSV_DIR / f"{row.run}.krona.tsv"
    output_html = RES_KRONA_HTML_DIR / f"{row.run}.krona.html"
    kraken_report_to_krona_text(row.kraken_report, krona_input)
    assert krona_input.stat().st_size > 0, f"Input Krona kosong: {krona_input}"
    run(["ktImportText", krona_input, "-o", output_html])
    krona_files.append(str(output_html))

samples["krona_html"] = krona_files
print("✅ Krona selesai diproduksi untuk seluruh sampel.")

In [ ]:
# @title 6.2 Tampilkan Krona untuk satu sampel contoh (V01)
import base64

v01_krona = Path(samples.loc[samples["sample"] == "V01", "krona_html"].iloc[0])
encoded = base64.b64encode(v01_krona.read_bytes()).decode("ascii")
display(HTML(f'<iframe src="data:text/html;base64,{encoded}" width="100%" height="650" style="border:1px solid #ddd; border-radius:6px;"></iframe>'))
print("💡 Tips: Jika diagram tidak tampil interaktif di browser Anda, unduh file ini dari panel Files kiri lalu buka di Chrome/Safari:", v01_krona)


### Cara membaca Krona Plot

* **Lingkaran paling dalam** menunjukkan tingkat **Domain dan Filum** (*Actinomycetota* / dulu *Actinobacteria*, *Bacteroidota* / *Bacteroidetes*, *Bacillota* / *Firmicutes* mengikuti tata nama resmi **ICNP 2021**).
* **Makin ke luar**, rinciannya makin spesifik ke **Family, Genus, hingga Spesies**. Klik sektor mana saja untuk memperbesar (*zoom-in*) dan melihat proporsinya.

---
## 7. Ekologi komunitas (Diversity Analysis)

Sebelum membandingkan keanekaragaman antar-sampel, kedalaman sekuensing perlu disamakan dulu lewat **rarefaction** agar sampel yang punya lebih banyak read tidak otomatis tampak lebih kaya takson.

> **Kenapa rarefaction dan uji diversity di sini dijalankan pada matriks Genus?**  
> Pada tabel diagnostik Sel 5.1, matriks **Genus** memberi kedalaman read yang jauh lebih besar dan merata lintas 12 sampel (minimum >12.000 read/sampel) dengan keyakinan taksonomi yang kuat pada data R9.4.1. Sementara itu, temuan tingkat **Spesies** tetap kita tampilkan di Sel 5.2 dan Sel 7.3.

In [ ]:
# @title 7.1 Rarefaction, 5 indeks alpha diversity (Genus), dan perbandingan kelompok
import numpy as np
from scipy.stats import mannwhitneyu

depths = genus_counts.sum(axis=0)
RAREFACTION_DEPTH = int(depths.min())
assert RAREFACTION_DEPTH > 0, "Kedalaman genus nol."
print(f"Kedalaman rarefaction genus: {RAREFACTION_DEPTH:,} reads/sampel")

rng = np.random.default_rng(SEED)
rarefied = pd.DataFrame(index=genus_counts.index)
for sample in genus_counts.columns:
    counts = genus_counts[sample].to_numpy(dtype=np.int64)
    rarefied[sample] = rng.multivariate_hypergeometric(counts, RAREFACTION_DEPTH)
assert (rarefied.sum(axis=0) == RAREFACTION_DEPTH).all()
rarefied.to_csv(RES_ALPHA_DIR / "genus_counts_rarefied.tsv", sep="\t")

def alpha_metrics(counts):
    counts = np.asarray(counts)
    counts = counts[counts > 0]
    s = len(counts)
    p = counts / counts.sum()
    shannon = -np.sum(p * np.log(p))
    pielou = shannon / np.log(s) if s > 1 else 0.0
    simpson_d = np.sum(p ** 2)
    return {
        "Observed genera": s,
        "Pielou evenness (J')": pielou,
        "Shannon (H')": shannon,
        "Simpson (1-D)": 1.0 - simpson_d,
        "Inverse Simpson (1/D)": 1.0 / simpson_d,
    }

alpha = pd.DataFrame({sample: alpha_metrics(rarefied[sample]) for sample in rarefied}).T
alpha.index.name = "sample"
alpha = alpha.reset_index().merge(samples[["sample", "group", "run"]], on="sample")
alpha.to_csv(RES_ALPHA_DIR / "alpha_diversity_genus.tsv", sep="\t", index=False)
display(alpha.round(3))

metrics = [
    "Observed genera",
    "Pielou evenness (J')",
    "Shannon (H')",
    "Simpson (1-D)",
    "Inverse Simpson (1/D)",
]
test_rows = []
for metric in metrics:
    vaginal = alpha.loc[alpha.group == "Vaginal", metric]
    sesar = alpha.loc[alpha.group == "Sesar", metric]
    statistic, p_value = mannwhitneyu(vaginal, sesar, alternative="two-sided")
    test_rows.append({
        "metric": metric,
        "median_vaginal": vaginal.median(),
        "median_sesar": sesar.median(),
        "U": statistic,
        "p_value": p_value,
    })

alpha_tests = pd.DataFrame(test_rows)
alpha_tests.to_csv(RES_ALPHA_DIR / "alpha_group_tests.tsv", sep="\t", index=False)
display(alpha_tests.round(4))

fig, axes = plt.subplots(1, len(metrics), figsize=(18, 4))
for ax, metric in zip(axes, metrics):
    sns.boxplot(data=alpha, x="group", y=metric, hue="group", palette=GROUP_COLORS, legend=False, width=0.5, ax=ax)
    sns.stripplot(data=alpha, x="group", y=metric, color="#222222", size=5.5, ax=ax)
    p_val = alpha_tests.loc[alpha_tests.metric == metric, "p_value"].iloc[0]
    ax.set_title(f"{metric}\nMann–Whitney p = {p_val:.4f}", fontsize=10.5)
    ax.set_xlabel("")
plt.tight_layout()
plt.savefig(RES_ALPHA_DIR / "alpha_diversity_boxplot.png", dpi=300)
plt.show()

### Panduan membaca 5 indeks Alpha Diversity

**Rarefaction** bisa dibayangkan seperti menyamakan jumlah tarikan jaring saat membandingkan jenis ikan di dua kolam. Semua sampel disamakan ke kedalaman minimum (`SEED = 42`) supaya perbandingannya adil.

#### Patokan praktis (*Rule of Thumb*) dan alasan menghitung tiap indeks

| Indeks | Rentang | Jika nilainya tinggi | Jika nilainya rendah | Kenapa perlu dihitung? |
|---|---|---|---|---|
| **Observed Genera** | `1 – ratusan` | Banyak jenis genus hadir (bayi 3 bln wajar 15–40; dewasa 80–150+). | Hanya sedikit jenis genus terdeteksi. | Mengukur **kekayaan murni (*Richness*)**: ada berapa banyak jenis bakteri. |
| **Pielou Evenness ($J'$)** | `0 – 1` | Mendekati `1`: porsi antar-bakteri merata. | Mendekati `0`: 1–2 bakteri mendominasi sampel. | Mengukur **kemerataan murni (*Evenness*)** secara terpisah dari jumlah jenis. |
| **Shannon ($H'$)** | `0 – ~4+` | Jenis banyak dan merata (`> 2,5` tinggi; `1–2` moderat/bayi). | Sedikit jenis atau ada monopoli kuat (`< 1,0`). | Indeks paling umum di jurnal klinis; menggabungkan kekayaan dan kemerataan. |
| **Simpson ($1 - D$)** | `0 – 1` | Mendekati `1` (`0,7–0,9`): peluang 2 read acak beda genus besar. | Mendekati `0` (`< 0,4`): 1 genus mendominasi total. | Lebih peka terhadap perubahan **bakteri dominan**, tidak mudah terpengaruh bakteri langka. |
| **Inverse Simpson ($1/D$)** | `1 – S` | Makin besar = makin banyak genus dominan setara (mis. `4,0` $\approx$ 4 genus dominan merata). | Mendekati `1`: hanya 1 genus yang dominan. | Tidak cepat jenuh di angka `0,99` seperti `1 - D`, dan angkanya mudah dibayangkan secara fisik. |

> **Analogi dua keranjang buah:** Keranjang A dan B sama-sama berisi 5 jenis buah (*Observed* = 5). Keranjang A isinya 96 apel dan masing-masing 1 jeruk, mangga, pisang, anggur (*Pielou* rendah). Keranjang B isinya 20 untuk tiap jenis buah (*Pielou* = 1,0 dan *Inverse Simpson* = 5,0).

#### Hal yang sering bikin bingung saat membaca jurnal
1. **Arah `Simpson D` vs `Simpson (1 - D)`**: di beberapa jurnal, penulis melaporkan **`Simpson D` (dominansi)** yang arahnya terbalik (makin besar = makin tidak beragam). Di notebook ini, kelima indeks sengaja dibuat searah: **makin tinggi = makin beragam/merata**.
2. **Kenapa `Chao1` dan `Faith's PD` tidak dihitung di sini?**
   * `Chao1` mengandalkan jumlah *singleton* (takson dengan tepat 1 read). Karena kita sudah membuang *noise* `< 10` read di matriks Genus, jumlah *singleton*-nya nol sehingga `Chao1` tidak lagi valid.
   * `Faith's PD` memerlukan pohon filogenetik hasil *multiple sequence alignment*, sedangkan `Kraken2` bekerja berbasis *k-mer* tanpa membuat pohon.
3. **Diversity tinggi tidak selalu berarti lebih sehat**: pada bayi ASI usia 3 bulan, Alpha Diversity yang rendah–moderat sering kali normal dan sehat karena ususnya memang didominasi *Bifidobacterium* pencerna ASI. Hal serupa juga berlaku pada mikrobioma vagina sehat yang didominasi *Lactobacillus*.

#### Temuan dari 12 sampel kita (Sel 7.1)
* Median **Inverse Simpson ($1/D$)** berada di kisaran **`~3,97` (Vaginal) vs `~3,39` (Sesar)** dan **Pielou ($J'$)** sekitar **`~0,50` vs `~0,46`**. Artinya, komunitas usus bayi 3 bulan ini setara dengan sekitar 3,5–4 genus dominan yang terbagi cukup merata.
* Semua nilai $p$ Mann–Whitney $> 0,05$. Jadi jumlah jenis dan kemerataan *di dalam* masing-masing sampel relatif sebanding antar-kelompok (di samping ukuran sampel memang masih kecil, $n=6$/grup). Untuk melihat apakah *jenis* bakterinya yang berbeda, kita cek **Beta Diversity** di sel berikutnya.

In [ ]:
# @title 7.2 Beta Diversity: Bray–Curtis (Kelimpahan), Jaccard (Ada/Tidak), PCoA, & PERMANOVA
from scipy.spatial.distance import pdist, squareform

sample_by_genus = rarefied.T.loc[samples["sample"]]

# 1. Bray-Curtis (memperhitungkan proporsi/kelimpahan read)
distance_bc = squareform(pdist(sample_by_genus, metric="braycurtis"))
beta_bc = pd.DataFrame(distance_bc, index=sample_by_genus.index, columns=sample_by_genus.index)
beta_bc.to_csv(RES_BETA_DIR / "bray_curtis_genus.tsv", sep="\t")

# 2. Jaccard (presence/absence: hanya melihat ada/tidaknya genus)
presence_genus = (sample_by_genus > 0).astype(int)
distance_jac = squareform(pdist(presence_genus, metric="jaccard"))
beta_jac = pd.DataFrame(distance_jac, index=sample_by_genus.index, columns=sample_by_genus.index)
beta_jac.to_csv(RES_BETA_DIR / "jaccard_genus.tsv", sep="\t")

# PCoA pada Bray-Curtis (classical multidimensional scaling)
n = len(beta_bc)
J = np.eye(n) - np.ones((n, n)) / n
B = -0.5 * J @ (distance_bc ** 2) @ J
eigenvalues, eigenvectors = np.linalg.eigh(B)
order = np.argsort(eigenvalues)[::-1]
eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:, order]
positive = eigenvalues > 1e-12
coords = eigenvectors[:, positive] * np.sqrt(eigenvalues[positive])
explained = eigenvalues[positive] / eigenvalues[positive].sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(beta_bc, vmin=0, vmax=1, cmap="Reds", square=True, ax=axes[0])
axes[0].set_title("Bray–Curtis dissimilarity (Genus)")

group_lookup = samples.set_index("sample")["group"]
for group, color in GROUP_COLORS.items():
    mask = np.array([group_lookup[s] == group for s in beta_bc.index])
    axes[1].scatter(coords[mask, 0], coords[mask, 1], s=80, label=group, color=color)
for label, x, y in zip(beta_bc.index, coords[:, 0], coords[:, 1]):
    axes[1].annotate(label, (x, y), xytext=(4, 4), textcoords="offset points", fontsize=9)
axes[1].set_xlabel(f"PCoA1 ({explained[0] * 100:.1f}%)")
axes[1].set_ylabel(f"PCoA2 ({explained[1] * 100:.1f}%)")
axes[1].set_title("PCoA Bray–Curtis (Tingkat Genus)")
axes[1].legend(title="Persalinan")
plt.tight_layout()
plt.savefig(RES_BETA_DIR / "beta_diversity_genus_pcoa.png", dpi=300)
plt.show()

def permanova_one_factor(distance_matrix, labels, permutations=999, seed=42):
    D = np.asarray(distance_matrix, dtype=float)
    labels = np.asarray(labels)
    n = len(labels)
    G = -0.5 * (np.eye(n) - np.ones((n, n))/n) @ (D ** 2) @ (np.eye(n) - np.ones((n, n))/n)
    groups = np.unique(labels)

    def pseudo_f(current_labels):
        X = np.column_stack([current_labels == g for g in groups]).astype(float)
        H = X @ np.linalg.pinv(X.T @ X) @ X.T
        ss_between = np.trace(H @ G)
        ss_total = np.trace(G)
        return (ss_between / (len(groups) - 1)) / ((ss_total - ss_between) / (n - len(groups)))

    observed = pseudo_f(labels)
    rng = np.random.default_rng(seed)
    permuted = np.array([pseudo_f(rng.permutation(labels)) for _ in range(permutations)])
    p_value = (1 + np.sum(permuted >= observed)) / (permutations + 1)
    return observed, p_value

labels = samples.set_index("sample").loc[beta_bc.index, "group"].to_numpy()
f_bc, p_bc = permanova_one_factor(distance_bc, labels, permutations=999, seed=SEED)
f_jac, p_jac = permanova_one_factor(distance_jac, labels, permutations=999, seed=SEED)

permanova_result = pd.DataFrame([
    {"metric": "Bray-Curtis (abundance)", "pseudo_F": f_bc, "p_value": p_bc, "permutations": 999, "n_per_group": 6},
    {"metric": "Jaccard (presence/absence)", "pseudo_F": f_jac, "p_value": p_jac, "permutations": 999, "n_per_group": 6},
])
permanova_result.to_csv(RES_BETA_DIR / "permanova_genus.tsv", sep="\t", index=False)
display(permanova_result.round(4))

### Panduan membaca Beta Diversity (Bray–Curtis, Jaccard, PCoA, dan PERMANOVA)

Jika Alpha Diversity melihat keragaman *di dalam satu sampel*, Beta Diversity membandingkan *seberapa berbeda komposisi antar-sampel atau antar-kelompok*.

#### Patokan praktis (*Rule of Thumb*) dan alasan membandingkan Bray–Curtis vs Jaccard

| Metrik | Rentang | Jika nilainya tinggi | Jika nilainya rendah | Kenapa perlu dihitung? |
|---|---|---|---|---|
| **Bray–Curtis** *(kelimpahan %)* | `0 – 1` | Mendekati `1` (`> 0,7`): komposisi sangat berbeda. | Mendekati `0` (`< 0,3`): komposisi sangat mirip. | Memperhitungkan proporsi (%) tiap bakteri, sehingga peka saat bakteri dominan bergeser. *Ingat: makin tinggi = makin beda.* |
| **Jaccard** *(ada / tidak ada)* | `0 – 1` | Mendekati `1`: daftar jenis bakteri berbeda jauh. | Mendekati `0`: daftar jenis bakteri hampir sama. | Memberi bobot setara bagi bakteri langka dan dominan. Jika Bray–Curtis tinggi tapi Jaccard rendah, jenis bakterinya mirip namun porsinya berubah jauh. |
| **PCoA (% di sumbu)** | `0–100%` | Total PCoA1 + PCoA2 `> 50%`: peta 2D cukup baik mewakili jarak asli. | `< 20%`: variasi data kompleks; peta 2D hanya menangkap sebagian pola. | Seperti peta jarak antar-kota: titik berdekatan berarti komposisinya mirip, berjauhan berarti beda. |
| **PERMANOVA (`Pseudo-F`, `p`)** | `F > 0`, `p: 0–1` | `F` besar dan `p < 0,05`: pemisahan antar-kelompok melampaui variasi acak internal. | `p ≥ 0,05`: belum cukup bukti statistik terpisah (bisa karena mirip atau $n$ kecil). | Selalu baca bersama plot PCoA karena `p < 0,05` bisa disebabkan perbedaan pusat (*centroid*) maupun sebaran (*dispersi*). |

> **Catatan tentang `UniFrac` di jurnal lain:** Prinsip *Weighted* dan *Unweighted UniFrac* mirip dengan Bray–Curtis dan Jaccard, hanya saja UniFrac menambahkan jarak evolusi dari pohon filogenetik (tidak dihitung di sini karena Kraken2 tidak menghasilkan pohon).

#### Temuan dari 12 sampel kita (Sel 7.2)
* Pada tabel PERMANOVA, **`Bray–Curtis` (`Pseudo-F ≈ 1,71`, `p ≈ 0,13`) lebih besar dibanding `Jaccard` (`Pseudo-F ≈ 1,09`, `p ≈ 0,35`)**.
* Ini menunjukkan banyak genus sebenarnya sama-sama hadir di kedua kelompok, tetapi **proporsi genus dominannya** (*Bacteroides*, *Parabacteroides*, *Phocaeicola* di Vaginal vs *Clostridium*, *Veillonella* di Sesar) yang paling terasa bergeser.

### 7.3 Takson yang berbeda antar-kelompok (Uji Hipotesis Eksploratif)

Kita membandingkan kelimpahan relatif setiap **Genus** antara kelompok persalinan Vaginal vs Sesar menggunakan uji non-parametrik Mann–Whitney U, lalu mengoreksi pengujian hipotesis multipel dengan metode **Benjamini–Hochberg False Discovery Rate (FDR)**. Di bawahnya, kita juga menampilkan ringkasan eksploratif deteksi **Spesies terarah (*putative species*)** lintas kedua kelompok.


In [ ]:
# @title 7.3 Uji Genus (Mann–Whitney + BH-FDR) & Ringkasan Eksplorasi Spesies
from scipy.stats import mannwhitneyu

def benjamini_hochberg(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    ranked = p_values[order]
    adjusted_ranked = np.minimum.accumulate((ranked * len(ranked) / np.arange(1, len(ranked)+1))[::-1])[::-1]
    adjusted = np.empty_like(adjusted_ranked)
    adjusted[order] = np.clip(adjusted_ranked, 0, 1)
    return adjusted

relative_rarefied = rarefied.div(rarefied.sum(axis=0), axis=1)
v_samples = samples.loc[samples.group == "Vaginal", "sample"].tolist()
c_samples = samples.loc[samples.group == "Sesar", "sample"].tolist()

# 1. Uji Statistik Utama: Tingkat Genus
rows = []
for genus in relative_rarefied.index:
    v = relative_rarefied.loc[genus, v_samples]
    c = relative_rarefied.loc[genus, c_samples]
    statistic, p_value = mannwhitneyu(v, c, alternative="two-sided")
    rows.append({
        "genus": genus,
        "median_vaginal_pct": 100 * v.median(),
        "median_sesar_pct": 100 * c.median(),
        "prevalence_vaginal": (v > 0).mean(),
        "prevalence_sesar": (c > 0).mean(),
        "U": statistic,
        "p_value": p_value,
    })
genus_tests = pd.DataFrame(rows)
genus_tests["q_value_BH"] = benjamini_hochberg(genus_tests["p_value"])
genus_tests = genus_tests.sort_values(["q_value_BH", "p_value"])
genus_tests.to_csv(RES_DIFF_DIR / "genus_group_tests.tsv", sep="\t", index=False)
print("=== Uji Perbedaan Kelimpahan Tingkat Genus (Koreksi BH-FDR) ===")
display(genus_tests.head(15).round(4))
print("Jumlah genus dengan q < 0.05 (FDR):", int((genus_tests.q_value_BH < 0.05).sum()))

# 2. Ringkasan Eksploratif: Spesies Terarah (Putative Species)
if not species_counts.empty:
    sp_rows = []
    for sp in species_counts.index:
        v_cnt = species_counts.loc[sp, v_samples]
        c_cnt = species_counts.loc[sp, c_samples]
        sp_rows.append({
            "species": sp,
            "total_reads_vaginal": int(v_cnt.sum()),
            "total_reads_sesar": int(c_cnt.sum()),
            "prevalence_vaginal": (v_cnt > 0).mean(),
            "prevalence_sesar": (c_cnt > 0).mean(),
        })
    species_summary = pd.DataFrame(sp_rows).sort_values(
        by=["total_reads_vaginal", "total_reads_sesar"], ascending=False
    )
    species_summary.to_csv(RES_DIFF_DIR / "species_group_summary.tsv", sep="\t", index=False)
    print("\n=== Ringkasan Eksplorasi Spesies Terarah (Putative Species, NCBI Targeted Loci) ===")
    display(species_summary.head(15).round(3))


### Memahami `q_value_BH` (FDR) dan batas interpretasi spesies

* **Kenapa perlu melihat `q_value_BH`, bukan `p_value` mentah?** Ketika kita menguji puluhan genus sekaligus, sebagian bisa menghasilkan $p < 0,05$ hanya karena kebetulan (*multiple testing*). Koreksi **Benjamini–Hochberg (FDR)** mengubahnya menjadi $q$-value (dianggap bermakna bila $q < 0,05$).
* **Batas klinis**: tabel spesies di atas membantu eksplorasi biologis, tetapi **deteksi DNA 16S tidak otomatis berarti bakteri hidup atau infeksi klinis**.

#### Temuan dari 12 sampel kita (Sel 7.3)
* **Efek koreksi BH-FDR**: ***Parabacteroides*** dan ***Phocaeicola*** memiliki $p = 0,0284$ (terlihat signifikan jika melihat $p$ mentah), tetapi setelah koreksi Benjamini–Hochberg pada $n=6$/grup menjadi $q = 0,632$. Ini contoh nyata mengapa kita tidak boleh langsung mengklaim signifikansi pada sampel kecil tanpa koreksi multipel.
* **Pola spesies terarah (*putative species*) yang menonjol**:
  * **Banyak di kelompok Vaginal**: *Bacteroides fragilis* (~17.772 vs 12 read), *Parabacteroides distasonis* (~17.464 vs 2 read), *Phocaeicola dorei* (~12.655 vs 2 read), dan *Phocaeicola vulgatus* (~2.111 vs 6 read) — sejalan dengan transmisi flora usus/vaginal maternal.
  * **Kuat di kedua kelompok**: *Bifidobacterium longum* (~9.172 vs 6.399 read, hadir di 100% sampel) — bakteri komensal pencerna oligosakarida ASI (HMO).
  * **Lebih tinggi di kelompok Sesar**: *Clostridium neonatale* (~7.911 vs 6.165 read).
* **Trade-off `--confidence 0.01`**: di bagian bawah tabel kadang muncul 1–2 spesies yang tidak lazim pada usus bayi (misalnya *Holdemanella porci* atau *Kluyvera ascorbata*). Ini contoh *false positive* ringan dari pencocokan *k-mer* pada data R9.4.1, dan alasan mengapa studi riset tingkat spesies biasanya dilanjutkan dengan **`Emu`** (Curry et al., 2022).

## 8. Interpretasi yang bertanggung jawab

Biasakan melaporkan hasil dengan urutan: **observasi → angka → ketidakpastian → batas kesimpulan**.

Hindari kalimat kesimpulan seperti ini:
- “Sesar menyebabkan genus X hilang” — studi ini observasional, bukan eksperimen intervensi.
- “p > 0,05 membuktikan kedua kelompok sama” — dengan 6 sampel per kelompok, daya statistiknya memang terbatas.
- “PERMANOVA signifikan berarti pusat kelompok pasti berbeda” — perbedaan sebaran (*dispersi*) juga bisa memicu nilai signifikan.
- “Genus X adalah patogen” — data 16S tidak melihat galur (*strain*), gen toksin, viabilitas sel, ataupun gejala klinis.
- “Persentase naik berarti jumlah sel bertambah” — data 16S bersifat komposisional dan dipengaruhi bias PCR serta jumlah kopi gen 16S.

### Pertanyaan diskusi
1. Apakah persentase read yang lolos filter sebanding antara kedua kelompok?
2. Apakah boxplot Alpha Diversity banyak tumpang tindih antara Vaginal dan Sesar?
3. Apakah sampel mengelompok di plot PCoA, dan apakah ada sampel yang menyimpang jauh (*outlier*)?
4. Apakah ada genus yang tetap signifikan setelah koreksi BH-FDR, dan apakah pola spesiesnya sejalan?
5. Faktor perancu (*confounder*) apa saja yang perlu dicatat pada penelitian penuh?
6. Mengapa subset 6+6 ini tidak boleh dianggap mereproduksi seluruh hasil publikasi aslinya?

### Contoh ringkasan laporan 4 langkah untuk dataset kita

> 1. **Observasi**: Dari 12 sampel feses bayi usia 3 bulan (PRJNA637202, amplikon V3–V9 ~1,14 kb, Nanopore R9.4.1) yang difilter `Chopper` (retensi ~70–72%) dan diklasifikasikan dengan `Kraken2` (`--confidence 0.01`, database NCBI Targeted Loci), ditemukan 89 genus ($\ge 10$ read) dan 150 spesies terarah ($\ge 10$ read).
> 2. **Angka**: Setelah rarefaction 12.460 read, Alpha Diversity relatif sebanding (*Inverse Simpson* ~3,97 vs 3,39; $p = 0,70$). Pada komposisi, `Bray–Curtis` (`F ≈ 1,71`) lebih memisahkan kelompok dibanding `Jaccard` (`F ≈ 1,09`), dengan pengayaan *Bacteroides* (*B. fragilis*), *Parabacteroides* (*P. distasonis*), dan *Phocaeicola* di kelompok Vaginal ($p$ mentah = 0,0284; $q_{\text{BH}} = 0,632$).
> 3. **Ketidakpastian**: Jumlah sampel kecil ($n=6$/grup), error indel R9.4.1 membatasi ketepatan *k-mer* tingkat spesies, dan kovariat penting (pola menyusui ASI/formula, antibiotik ibu) belum dikontrol.
> 4. **Batas kesimpulan**: Pola ini merupakan asosiasi eksploratif yang sejalan dengan literatur kolonisasi awal usus bayi, bukan bukti sebab-akibat langsung ataupun dasar diagnosis klinis.

---
## 9. Simpan hasil (Terorganisir per Tahap Analisis)

Seluruh berkas keluaran telah disusun ke dalam subfolder bernomor agar mudah ditelusuri (*traceability*) dan langsung siap dilampirkan sebagai suplementasi riset:

```text
results/
├── 01_qc/                      # Statistik SeqKit, qc_summary.tsv, & laporan NanoPlot
├── 02_classification/          # classification_summary.tsv, reports/ (.k2report), & per_read/ (.kraken.tsv)
├── 03_abundance/               # tables/ (matriks Genus & Spesies + LCA depth) & figures/ (barplot PNG)
├── 04_krona/                   # html/ (Krona interaktif) & tsv/ (input hierarki)
├── 05_diversity/               # alpha/ (rarefaction, Shannon/Simpson, boxplot) & beta/ (Bray-Curtis, PCoA, PERMANOVA)
├── 06_differential_abundance/  # Uji Mann-Whitney + BH-FDR (Genus) & ringkasan Spesies terarah
└── run_manifest.json           # Metadata jejak audit (timestamp, parameter, SHA-256 database, daftar sampel)
```

*(FASTQ mentah & terfilter berada di luar `results/` sehingga arsip ZIP tetap ringkas.)*


In [ ]:
# @title 9.1 Catat run_manifest.json, arsipkan hasil terorganisir, dan unduh ZIP
import json
from datetime import datetime, timezone

manifest_meta = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": "PRJNA637202 (PreventADALL; Angell et al., 2020, PeerJ)",
    "database": {
        "name": "ONT EPI2ME NCBI Targeted Loci 16S/18S",
        "url": DB_URL,
        "expected_sha256": EXPECTED_SHA256,
    },
    "parameters": {
        "min_length": MIN_LENGTH,
        "max_length": MAX_LENGTH,
        "min_mean_q": MIN_MEAN_Q,
        "kraken_confidence": KRAKEN_CONFIDENCE,
        "rarefaction_depth_genus": int(RAREFACTION_DEPTH),
        "seed": SEED,
        "n_samples": int(len(samples)),
    },
    "samples": samples[["sample", "group", "run"]].to_dict(orient="records"),
}
manifest_path = RESULTS_DIR / "run_manifest.json"
manifest_path.write_text(json.dumps(manifest_meta, indent=2), encoding="utf-8")

print("📂 Ringkasan struktur folder results/:")
for sub in sorted(p for p in RESULTS_DIR.rglob("*") if p.is_dir()):
    n_files = sum(1 for f in sub.iterdir() if f.is_file())
    if n_files > 0:
        print(f"   • {sub.relative_to(RESULTS_DIR)}/ ({n_files} file)")

archive_base = WORKDIR / "16s_workshop_results"
archive = shutil.make_archive(str(archive_base), "zip", root_dir=WORKDIR, base_dir="results")
print(f"\n✅ Arsip siap: {archive} ({Path(archive).stat().st_size / 1e6:.2f} MB)")

try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Bukan Colab; ambil file secara manual.")


## Checklist sebelum melaporkan hasil

- [ ] Accession, barcode, alias ENA, dan kelompok sampel dicatat.
- [ ] Versi alat (`kraken2`, `chopper`), database (`NCBI Targeted Loci 16S/18S`), parameter, dan seed dicatat.
- [ ] Retensi filtering diperiksa.
- [ ] Classified/unclassified dilaporkan.
- [ ] Kedalaman distandardisasi via rarefaction sebelum perbandingan diversity (Genus) & eksplorasi Spesies.
- [ ] Keterbatasan resolusi spesies pada gen 16S dan potensi bias primer dibahas.
- [ ] Multiple testing dikoreksi (Benjamini–Hochberg FDR) dan ukuran sampel dilaporkan.
- [ ] Hasil diperlakukan sebagai asosiasi eksploratif, bukan efek kausal atau diagnosis klinis.

**Sumber utama data:** ENA/NCBI BioProject PRJNA637202 (Angell et al., 2020). **Database workshop:** ONT EPI2ME NCBI Targeted Loci 16S/18S Kraken2 archive. Untuk laporan ilmiah, cantumkan sitasi resmi versi alat dan database yang digunakan.